In [225]:
import pandas as pd

rsd_path = "../../output/rsd/rolling_stock.xlsx"
notification_path = "../../output/notification_workorder.xlsx"

# Load all sheets
dfs_rsd = pd.read_excel(rsd_path, sheet_name=None, keep_default_na=False)
dfs_notif = pd.read_excel(notification_path, sheet_name=None)

### Tyre Pressure

In [226]:
df_notification = dfs_notif.get("notification")

df_notification = df_notification[
    df_notification["maint_work_centre"].astype(str).str.strip().eq("RSDM")
]

df_notification = df_notification[
    ["notification_no", "work_order_no", "functional_location", "work_request", "target_date"]
].copy()

df_notification.rename(columns={
    "notification_no": "notification_id",
    "work_order_no": "workorder_id",
}, inplace=True)

df_notification["functional_location_code"] = df_notification["functional_location"].str.split(" ").str[0]
df_notification["work_request_code"] = df_notification["work_request"].str.extract(r"\((.*?)\)")

# drop original column replace with code column
df_notification.drop(columns=["functional_location", "work_request"], inplace=True)
df_notification.rename(columns={
    "functional_location_code": "functional_location",
    "work_request_code": "work_request"
}, inplace=True)

df_notification.head()

,notification_id,workorder_id,target_date,functional_location,work_request
0,11808988,4000410420,02/07/2021,RSV021,WEK1
1,11886564,4000441582,26/12/2021,RSV023,WEK2
4,11887904,4000442018,30/12/2021,RSV025,WEK2
6,11887974,4000441786,31/12/2021,RSV021,WEK3
9,11888391,4000442299,01/01/2022,RSV022,WEK3


In [227]:
df_tyre_pressure = dfs_rsd.get("tyre_pressure")
df_tyre_wear = dfs_rsd.get("tyre_wear")

for df in [df_tyre_pressure, df_tyre_wear]:
    if df is not None:
        df.columns = df.columns.str.strip()

df_tyre_pressure = df_tyre_pressure.add_prefix("tyre_pressure.")
df_tyre_wear = df_tyre_wear.add_prefix("tyre_wear.")

bogie_sn_cols = [c for c in df_tyre_pressure.columns if c.endswith("bogie_sn")]

df_tyre_pressure.rename(columns={
    "tyre_pressure.workorder_id": "workorder_id",
    "tyre_pressure.filename": "filename",
    "tyre_pressure.approval.date": "tyre_pressure.approval_date",
    "tyre_pressure.approval.technician_id": "tyre_pressure.technician_id",
    "tyre_pressure.approval.supervisor_id": "tyre_pressure.supervisor_id",
}, inplace=True)

df_tyre_pressure[bogie_sn_cols] = (
    df_tyre_pressure[bogie_sn_cols]
        .replace(r"^\s*$", pd.NA, regex=True)    # blank / whitespace -> NA
        .apply(pd.to_numeric, errors="coerce")   # invalid -> NA
        .astype("Int64")                         # force Int64
)

df_tyre_wear.rename(columns={
    "tyre_wear.workorder_id": "workorder_id",
    "tyre_wear.filename": "filename",
    "tyre_wear.approval.date": "tyre_wear.approval_date",
    "tyre_wear.approval.technician_id": "tyre_wear.technician_id",
    "tyre_wear.approval.supervisor_id": "tyre_wear.supervisor_id",
}, inplace=True)

In [228]:
pd.set_option("display.max_rows", None)

print(df_tyre_pressure.isna().sum())


tyre_pressure.eca1.bogie1.bogie_sn                       2
tyre_pressure.eca1.bogie1.top_guide_wheel.a.before       0
tyre_pressure.eca1.bogie1.top_guide_wheel.a.after        0
tyre_pressure.eca1.bogie1.top_guide_wheel.b.before       0
tyre_pressure.eca1.bogie1.top_guide_wheel.b.after        0
tyre_pressure.eca1.bogie1.top_guide_wheel.c.before       0
tyre_pressure.eca1.bogie1.top_guide_wheel.c.after        0
tyre_pressure.eca1.bogie1.load_wheel.before              0
tyre_pressure.eca1.bogie1.load_wheel.after               0
tyre_pressure.eca1.bogie1.bottom_guide_wheel.a.before    0
tyre_pressure.eca1.bogie1.bottom_guide_wheel.a.after     0
tyre_pressure.eca1.bogie1.bottom_guide_wheel.b.before    0
tyre_pressure.eca1.bogie1.bottom_guide_wheel.b.after     0
tyre_pressure.eca1.bogie1.bottom_guide_wheel.c.before    0
tyre_pressure.eca1.bogie1.bottom_guide_wheel.c.after     0
tyre_pressure.eca1.bogie2.bogie_sn                       2
tyre_pressure.eca1.bogie2.top_guide_wheel.a.before      

In [229]:
pd.set_option("display.max_rows", None)

print(df_tyre_wear.isna().sum())


tyre_wear.eca1.bogie1.bogie_sn                        0
tyre_wear.eca1.bogie1.top_guide_wheel.a.groove1       0
tyre_wear.eca1.bogie1.top_guide_wheel.a.groove5       0
tyre_wear.eca1.bogie1.top_guide_wheel.b.groove1       0
tyre_wear.eca1.bogie1.top_guide_wheel.b.groove5       0
tyre_wear.eca1.bogie1.top_guide_wheel.c.groove1       0
tyre_wear.eca1.bogie1.top_guide_wheel.c.groove5       0
tyre_wear.eca1.bogie1.load_wheel.a.groove1            0
tyre_wear.eca1.bogie1.load_wheel.a.groove2            0
tyre_wear.eca1.bogie1.load_wheel.a.groove6            0
tyre_wear.eca1.bogie1.load_wheel.a.groove7            0
tyre_wear.eca1.bogie1.load_wheel.b.groove1            0
tyre_wear.eca1.bogie1.load_wheel.b.groove2            0
tyre_wear.eca1.bogie1.load_wheel.b.groove6            0
tyre_wear.eca1.bogie1.load_wheel.b.groove7            0
tyre_wear.eca1.bogie1.bottom_guide_wheel.a.groove1    0
tyre_wear.eca1.bogie1.bottom_guide_wheel.a.groove5    0
tyre_wear.eca1.bogie1.bottom_guide_wheel.b.groov

In [230]:
import pandas as pd

bogie_sn_cols = [c for c in df_tyre_pressure.columns if c.endswith("bogie_sn")]

df_bogie_sn = df_tyre_pressure[["filename", "workorder_id"] + bogie_sn_cols].copy()

# Clean + force ALL bogie_sn cols to Int64 only
df_bogie_sn[bogie_sn_cols] = (
    df_bogie_sn[bogie_sn_cols]
        .replace(r"^\s*$", pd.NA, regex=True)    # blank / whitespace -> NA
        .apply(pd.to_numeric, errors="coerce")   # invalid -> NA
        .astype("Int64")                         # force Int64
)

# Hard check: ensure no bogie_sn col is anything other than Int64
bad = {c: str(df_bogie_sn[c].dtype) for c in bogie_sn_cols if str(df_bogie_sn[c].dtype) != "Int64"}
if bad:
    raise TypeError(f"These bogie_sn columns are not Int64: {bad}")

df_bogie_sn.head()


,filename,workorder_id,tyre_pressure.eca1.bogie1.bogie_sn,tyre_pressure.eca1.bogie2.bogie_sn,tyre_pressure.ica2.bogie1.bogie_sn,tyre_pressure.ica2.bogie2.bogie_sn,tyre_pressure.ica3.bogie1.bogie_sn,tyre_pressure.ica3.bogie2.bogie_sn,tyre_pressure.eca4.bogie1.bogie_sn,tyre_pressure.eca4.bogie2.bogie_sn
0,RS_PM_MTH_4000590580.pdf,4000590580,264,235,217,299,212,278,222,257
1,RS_PM_WEK_4000410420.pdf,4000410420,251,232,225,226,217,219,255,201
2,RS_PM_WEK_4000441582.pdf,4000441582,254,211,223,212,213,206,220,253
3,RS_PM_WEK_4000441786.pdf,4000441786,244,203,225,226,217,214,204,218
4,RS_PM_WEK_4000442018.pdf,4000442018,231,249,223,201,224,235,221,219


In [231]:
df_bogie_sn.dtypes

filename                              object
workorder_id                           int64
tyre_pressure.eca1.bogie1.bogie_sn     Int64
tyre_pressure.eca1.bogie2.bogie_sn     Int64
tyre_pressure.ica2.bogie1.bogie_sn     Int64
tyre_pressure.ica2.bogie2.bogie_sn     Int64
tyre_pressure.ica3.bogie1.bogie_sn     Int64
tyre_pressure.ica3.bogie2.bogie_sn     Int64
tyre_pressure.eca4.bogie1.bogie_sn     Int64
tyre_pressure.eca4.bogie2.bogie_sn     Int64
dtype: object

In [232]:
# Ensure required columns exist
required_cols = ["workorder_id", "functional_location", "work_request", "target_date"]
df_notification_subset = df_notification[required_cols]


# Merge functional_location into tyre_pressure
df_tyre_pressure_final = df_tyre_pressure.merge(
    df_notification_subset,
    on="workorder_id",
    how="left"
)

bogie_sn_cols = [c for c in df_tyre_pressure_final.columns if c.endswith("bogie_sn")]
df_tyre_pressure_final[bogie_sn_cols] = df_tyre_pressure_final[bogie_sn_cols]

cols = df_tyre_pressure_final.columns.tolist()

preferred_order = ["filename", "workorder_id", "functional_location", "work_request", "target_date"]
remaining_cols = [c for c in cols if c not in preferred_order]

df_tyre_pressure_final = df_tyre_pressure_final[
    preferred_order + remaining_cols
]

df_tyre_pressure_final.head()


,filename,workorder_id,functional_location,work_request,target_date,tyre_pressure.eca1.bogie1.bogie_sn,tyre_pressure.eca1.bogie1.top_guide_wheel.a.before,tyre_pressure.eca1.bogie1.top_guide_wheel.a.after,tyre_pressure.eca1.bogie1.top_guide_wheel.b.before,tyre_pressure.eca1.bogie1.top_guide_wheel.b.after,...,tyre_pressure.eca4.bogie2.load_wheel.after,tyre_pressure.eca4.bogie2.bottom_guide_wheel.a.before,tyre_pressure.eca4.bogie2.bottom_guide_wheel.a.after,tyre_pressure.eca4.bogie2.bottom_guide_wheel.b.before,tyre_pressure.eca4.bogie2.bottom_guide_wheel.b.after,tyre_pressure.eca4.bogie2.bottom_guide_wheel.c.before,tyre_pressure.eca4.bogie2.bottom_guide_wheel.c.after,tyre_pressure.approval_date,tyre_pressure.technician_id,tyre_pressure.supervisor_id
0,RS_PM_MTH_4000590580.pdf,4000590580,RSV027,MTH,19/03/2024,264,10.6,10.6,10.0,10.0,...,11.2,9.5,10.5,10.2,10.2,10.4,10.4,19/03/2024,21120,7306
1,RS_PM_WEK_4000410420.pdf,4000410420,RSV021,WEK1,02/07/2021,251,10.5,11,10.5,11,...,12.0,10.5,11,10.5,11,10.5,11,10/07/2021,10475,7196
2,RS_PM_WEK_4000441582.pdf,4000441582,RSV023,WEK2,26/12/2021,254,13,11.5,13.5,14.5,...,,14,14.5,13.5,14.5,13.5,14.5,07/01/2021,11515,7192
3,RS_PM_WEK_4000441786.pdf,4000441786,RSV021,WEK3,31/12/2021,244,12.1,12.1,12.2,12.2,...,12.8,10.8,10.8,10.8,10.8,11.7,11.7,08/01/2022,7270,7127
4,RS_PM_WEK_4000442018.pdf,4000442018,RSV025,WEK2,30/12/2021,231,13.4,14.5,13.0,14.5,...,11.5,14,14.5,13.5,14.5,13.5,14.5,06/1/2022,11515,7192


In [233]:
import pandas as pd

# Try parsing strictly as DD/MM/YYYY
parsed_dates = pd.to_datetime(
    df_tyre_pressure_final['tyre_pressure.approval_date'],
    format='%d/%m/%Y',
    errors='coerce'
)

# Rows where parsing failed but original value is not null/empty
invalid_mask = parsed_dates.isna() & df_tyre_pressure_final['tyre_pressure.approval_date'].notna()

df_invalid_dates = df_tyre_pressure_final.loc[
    invalid_mask,
    ['filename', 'tyre_pressure.approval_date']
]

# df_invalid_dates


In [234]:
print(
    df_tyre_pressure_final.loc[invalid_mask, 'tyre_pressure.approval_date']
    .astype(str)
    .unique()
)

['01/02/20' '09/01/22' '28/01/22' '25/02/22' '31/03/22' '06/04.2022' ''
 '26/4/22' '6-7-2022' '20/08/22' '2/9/22' '17/9/22' '07/10/12022'
 '03/10/22' '08/10/22' '14/10/2' '23-12-2022' '27-09-2022' '4/8/98'
 '13/11/22' '09/11/22' '6/12/22' '15/12/22' '18/12/22' '19/12/22'
 '22/10/22' '04/01/23' '8/06/23' '27/01/23' '23/01/23' '04/02/23'
 '30/1/20' '31/01/23' '09/02*2023' '05/02/23' '24/02/23' '8/3/23'
 '12/5/23' '21-4-23' '26/05/23' '20/03/23' '04/06/23' '5/6/23' '21/06/23'
 '29/6' '2/7/23' '04/07/23' '18/7' '3/8/2' '12/' '16/08/202)' '27/08/202)'
 '2-9-03' '4/2000' '10/09/23' '22/09/23' '23' '16/10/23' '27/10/23'
 '09/11/23' '07/11/202)' '10/12/23' '09/01/23' '26' '039/03/2024'
 '8.6.2023' '10.6/2024' '22/6/24' '11/07/202' '6/7/24' '9-7-2020'
 '21/08/24' '18-9-2024' '28/01/25' '19/02/25' '03/03/27' '03/04/25'
 '05/042025' '4-5-2025' '18/6/25' '30/06/25']


In [235]:
before_cols = [c for c in df_tyre_pressure_final.columns if c.endswith(".before")]
after_cols  = [c.replace(".before", ".after") for c in before_cols
               if c.replace(".before", ".after") in df_tyre_pressure_final.columns]

cols_to_check = before_cols + after_cols

invalid_records = []

for col in cols_to_check:
    series = df_tyre_pressure_final[col]

    # normalize text for checking
    normalized = series.astype(str).str.strip().str.upper()

    # treat 'NEW' as valid (exclude from invalid check)
    is_new = normalized == "NEW"

    # numeric conversion
    numeric = pd.to_numeric(series, errors="coerce")

    # invalid = not null, not NEW, but cannot convert to number
    mask_invalid = (
        series.notna() &
        (~is_new) &
        numeric.isna()
    )

    if mask_invalid.any():
        temp = df_tyre_pressure_final.loc[mask_invalid, ["filename", col]].copy()
        temp["column"] = col
        temp.rename(columns={col: "invalid_value"}, inplace=True)
        invalid_records.append(temp)

invalid_df = pd.concat(invalid_records, ignore_index=True) if invalid_records else pd.DataFrame()

# invalid_df


In [236]:
import pandas as pd
import numpy as np

before_cols = [c for c in df_tyre_pressure_final.columns if c.endswith(".before")]

for before in before_cols:
    after = before.replace(".before", ".after")
    
    if after not in df_tyre_pressure_final.columns:
        continue

    before_series = df_tyre_pressure_final[before]
    after_series  = df_tyre_pressure_final[after]

    mask_after_missing = (
        before_series.notna() &
        (
            after_series.isna() |
            (after_series.astype(str).str.strip() == "")
        )
    )

    df_tyre_pressure_final.loc[mask_after_missing, after] = (
        before_series[mask_after_missing]
    )

    mask_before_missing = (
        after_series.notna() &
        (
            before_series.isna() |
            (before_series.astype(str).str.strip() == "")
        )
    )

    df_tyre_pressure_final.loc[mask_before_missing, before] = (
        after_series[mask_before_missing]
    )


In [237]:
import pandas as pd
import numpy as np

pressure_cols = [
    c for c in df_tyre_pressure_final.columns
    if c.endswith(".before") or c.endswith(".after")
]

invalid_values = []

for col in pressure_cols:
    original = df_tyre_pressure_final[col]

    # attempt conversion
    converted = pd.to_numeric(original, errors="coerce")

    # rows where conversion failed but original had value
    mask_invalid = original.notna() & converted.isna()

    # collect invalid values (DO NOT change them)
    if mask_invalid.any():
        temp = df_tyre_pressure_final.loc[mask_invalid, ["filename"]].copy()
        temp["column"] = col
        temp["invalid_value"] = original[mask_invalid].values
        invalid_values.append(temp)

    # rows where conversion is successful → overwrite
    mask_valid = converted.notna()
    df_tyre_pressure_final.loc[mask_valid, col] = converted[mask_valid]

if invalid_values:
    df_invalid_values = pd.concat(invalid_values, ignore_index=True)
    print("❌ Values NOT converted to float:")
    df_invalid_values
else:
    print("✅ All pressure values successfully converted to float.")


❌ Values NOT converted to float:


In [238]:
# combine all .before columns in one list and find unique values
before_cols = [
    c for c in df_tyre_pressure_final.columns
    if c.endswith(".before")
]
unique_before_values = set()
for col in before_cols:
    unique_before_values.update(df_tyre_pressure_final[col].dropna().unique())
print(unique_before_values)

{'', 1.0, 1.4, 0.0, 4.9, 4.2, 6.5, 5.0, 8.3, 9.1, 10.6, 11.1, 10.5, 12.1, 13.0, 13.4, 13.5, 11.0, 10.0, 10.1, 14.0, 12.3, 13.1, 14.5, 16.3, 16.0, 4.0, 5.5, 19.64, 4.5, 6.0, 0.5, 7.5, 7.0, 40.0, 8.0, 8.5, 10.52, 9.0, 9.5, 50.0, 10.11, 11.5, 11.04, 12.0, 12.5, 11.02, 15.0, 3.0, 16.1, 16.4, 16.9, 16.6, 17.0, 3.5, 16.5, 90.1, 19.1, 100.5, 101.5, 1.9, 101.0, 103.0, 105.0, 106.0, 107.0, 110.4, 111.0, 110.0, 110.8, 114.0, 110.5, 120.0, 0.2, 130.0, 135.0, 140.0, 145.0, 4.4, 150.0, 6.4, 6.9, 7.4, 7.9, 8.4, 8.9, 9.9, 9.4, 9.01, 9.87, 9.15, 10.9, 10.4, 10.37, 10.01, 11.4, 11.9, 11.022, 11.01, 12.4, 12.9, 13.9, 2.3, 3.3, 3.8, 4.3, 1.3, 1.8, 5.3, 6.8, 6.3, 7.8, 7.3, 0.9, 8.8, 8.7, 8.2, 8.02, 9.7, 9.3, 9.2, 9.8, 10.05, 10.3, 10.2, 10.8, 10.7, 11.2, 11.3, 11.8, 11.7, 10.02, 12.8, 12.7, 12.2, 11.05, 11.45, 13.7, 13.2, 13.3, 13.8, 14.3, 14.2, 10.41, 3.7, 16.7, 4.7, 5.7, 6.7, 19.8, 6.2, 7.2, 7.7, 'NEW', 903.0, 4.6, 5.1, 6.6, 6.1, 7.6, 7.1, 1.1, 1.2, 8.1, 8.6, 1.7, 9.6, 3.6, 3.1, 10.35, 11.6, 12.6, 13.6,

In [239]:
df_tyre_pressure_final["tyre_pressure.eca1.bogie1.bogie_sn"].dtype

Int64Dtype()

### Tyre Wear

In [240]:
# Ensure required columns exist
required_cols = ["workorder_id", "functional_location", "work_request", "target_date"]
df_notification_subset = df_notification[required_cols].drop_duplicates()

# Merge functional_location into tyre_wear
df_tyre_wear_final = df_tyre_wear.merge(
    df_notification_subset,
    on="workorder_id",
    how="left"
)

# Optional: reorder columns (workorder_id, filename first)
cols = df_tyre_wear_final.columns.tolist()

preferred_order = ["filename", "workorder_id", "functional_location", "work_request", "target_date"]
remaining_cols = [c for c in cols if c not in preferred_order]

df_tyre_wear_final = df_tyre_wear_final[
    preferred_order + remaining_cols
]

df_tyre_wear_final.head()


,filename,workorder_id,functional_location,work_request,target_date,tyre_wear.eca1.bogie1.bogie_sn,tyre_wear.eca1.bogie1.top_guide_wheel.a.groove1,tyre_wear.eca1.bogie1.top_guide_wheel.a.groove5,tyre_wear.eca1.bogie1.top_guide_wheel.b.groove1,tyre_wear.eca1.bogie1.top_guide_wheel.b.groove5,...,tyre_wear.eca4.bogie2.load_wheel.b.groove7,tyre_wear.eca4.bogie2.bottom_guide_wheel.a.groove1,tyre_wear.eca4.bogie2.bottom_guide_wheel.a.groove5,tyre_wear.eca4.bogie2.bottom_guide_wheel.b.groove1,tyre_wear.eca4.bogie2.bottom_guide_wheel.b.groove5,tyre_wear.eca4.bogie2.bottom_guide_wheel.c.groove1,tyre_wear.eca4.bogie2.bottom_guide_wheel.c.groove5,tyre_wear.approval_date,tyre_wear.technician_id,tyre_wear.supervisor_id
0,RS_PM_WEK_4000410420.pdf,4000410420,RSV021,WEK1,02/07/2021,251,5,5,5,5,...,5.5,5,5,4,4,7,7,10/07/2021,7148,7196
1,RS_PM_WEK_4000441582.pdf,4000441582,RSV023,WEK2,26/12/2021,254,4,6,4,4,...,6,5,6,5,6,4,3,07/01/2022,"9435,6196",7192
2,RS_PM_WEK_4000441786.pdf,4000441786,RSV021,WEK3,31/12/2021,244,5.0,5,5,5,...,8.0,6,6,5,5,5,5,08/01/2022,"9031,11139",7127
3,RS_PM_WEK_4000442018.pdf,4000442018,RSV025,WEK2,30/12/2021,231,6.0,6.0,6.0,6.0,...,8,5.0,4.0,5.0,5.0,6.0,4.0,06/01/2024,"9435,11517",7192
4,RS_PM_WEK_4000442299.pdf,4000442299,RSV022,WEK3,01/01/2022,224,4.0,4.0,5.0,5.0,...,8.5,6.0,6.0,6.0,6.0,6.5,6.5,03/01/2022,"7202,7202",7196


In [241]:
df_bogie_sn_wear = df_bogie_sn.copy()

df_bogie_sn_wear.rename(
    columns=lambda c: c.replace("tyre_pressure.", "tyre_wear.")
    if c.endswith("bogie_sn") else c,
    inplace=True
)

common_bogie_cols = [
    col for col in df_bogie_sn_wear.columns
    if col != "filename" and col in df_tyre_wear_final.columns
]

df_tyre_wear_final_replaced = df_tyre_wear_final.copy()

bogie_cols_wear = [col for col in df_bogie_sn_wear.columns if col != "filename"]

df_tyre_wear_final_replaced.set_index("filename", inplace=True)
df_bogie_sn_wear_indexed = df_bogie_sn_wear.set_index("filename")

common_filenames = df_tyre_wear_final_replaced.index.intersection(df_bogie_sn_wear_indexed.index)

df_tyre_wear_final_replaced.loc[common_filenames, bogie_cols_wear] = \
    df_bogie_sn_wear_indexed.loc[common_filenames, bogie_cols_wear]

df_tyre_wear_final_replaced.reset_index(inplace=True)

df_tyre_wear_final_replaced.head()


,filename,workorder_id,functional_location,work_request,target_date,tyre_wear.eca1.bogie1.bogie_sn,tyre_wear.eca1.bogie1.top_guide_wheel.a.groove1,tyre_wear.eca1.bogie1.top_guide_wheel.a.groove5,tyre_wear.eca1.bogie1.top_guide_wheel.b.groove1,tyre_wear.eca1.bogie1.top_guide_wheel.b.groove5,...,tyre_wear.eca4.bogie2.load_wheel.b.groove7,tyre_wear.eca4.bogie2.bottom_guide_wheel.a.groove1,tyre_wear.eca4.bogie2.bottom_guide_wheel.a.groove5,tyre_wear.eca4.bogie2.bottom_guide_wheel.b.groove1,tyre_wear.eca4.bogie2.bottom_guide_wheel.b.groove5,tyre_wear.eca4.bogie2.bottom_guide_wheel.c.groove1,tyre_wear.eca4.bogie2.bottom_guide_wheel.c.groove5,tyre_wear.approval_date,tyre_wear.technician_id,tyre_wear.supervisor_id
0,RS_PM_WEK_4000410420.pdf,4000410420,RSV021,WEK1,02/07/2021,251,5,5,5,5,...,5.5,5,5,4,4,7,7,10/07/2021,7148,7196
1,RS_PM_WEK_4000441582.pdf,4000441582,RSV023,WEK2,26/12/2021,254,4,6,4,4,...,6,5,6,5,6,4,3,07/01/2022,"9435,6196",7192
2,RS_PM_WEK_4000441786.pdf,4000441786,RSV021,WEK3,31/12/2021,244,5.0,5,5,5,...,8.0,6,6,5,5,5,5,08/01/2022,"9031,11139",7127
3,RS_PM_WEK_4000442018.pdf,4000442018,RSV025,WEK2,30/12/2021,231,6.0,6.0,6.0,6.0,...,8,5.0,4.0,5.0,5.0,6.0,4.0,06/01/2024,"9435,11517",7192
4,RS_PM_WEK_4000442299.pdf,4000442299,RSV022,WEK3,01/01/2022,224,4.0,4.0,5.0,5.0,...,8.5,6.0,6.0,6.0,6.0,6.5,6.5,03/01/2022,"7202,7202",7196


In [242]:
print(list(df.columns))

['eca1.bogie1.bogie_sn', 'eca1.bogie1.top_guide_wheel.a.groove1', 'eca1.bogie1.top_guide_wheel.a.groove5', 'eca1.bogie1.top_guide_wheel.b.groove1', 'eca1.bogie1.top_guide_wheel.b.groove5', 'eca1.bogie1.top_guide_wheel.c.groove1', 'eca1.bogie1.top_guide_wheel.c.groove5', 'eca1.bogie1.load_wheel.a.groove1', 'eca1.bogie1.load_wheel.a.groove2', 'eca1.bogie1.load_wheel.a.groove6', 'eca1.bogie1.load_wheel.a.groove7', 'eca1.bogie1.load_wheel.b.groove1', 'eca1.bogie1.load_wheel.b.groove2', 'eca1.bogie1.load_wheel.b.groove6', 'eca1.bogie1.load_wheel.b.groove7', 'eca1.bogie1.bottom_guide_wheel.a.groove1', 'eca1.bogie1.bottom_guide_wheel.a.groove5', 'eca1.bogie1.bottom_guide_wheel.b.groove1', 'eca1.bogie1.bottom_guide_wheel.b.groove5', 'eca1.bogie1.bottom_guide_wheel.c.groove1', 'eca1.bogie1.bottom_guide_wheel.c.groove5', 'eca1.bogie2.bogie_sn', 'eca1.bogie2.top_guide_wheel.a.groove1', 'eca1.bogie2.top_guide_wheel.a.groove5', 'eca1.bogie2.top_guide_wheel.b.groove1', 'eca1.bogie2.top_guide_wheel.b

In [243]:
import pandas as pd
import numpy as np
import re

# Step 1: get all groove columns
groove_cols = [c for c in df_tyre_wear_final_replaced.columns if "groove" in c]

safe_words = [
    "NEW", "REPLACED", "REPLACE", "OUTSIDE",
    "REPLACED WORN OUT", "NEW G/W"
]

invalid_values = []

for col in groove_cols:
    original = df_tyre_wear_final_replaced[col]

    # --- Step 2a: fix values like 8-5, 4-0 → 8.5, 4.0 ---
    fixed = (
        original.astype(str)
        .str.strip()
        .str.replace(
            r"^(\d+)-(\d+)$",   # ONLY digit-digit pattern
            r"\1.\2",
            regex=True
        )
    )

    # write fixed values back
    df_tyre_wear_final_replaced[col] = fixed.replace("nan", np.nan)

    # --- Step 2b: convert safe words to value = 2 ---
    mask_safe_word = (
        df_tyre_wear_final_replaced[col]
        .astype(str)
        .str.upper()
        .isin([w.upper() for w in safe_words])
    )

    df_tyre_wear_final_replaced.loc[mask_safe_word, col] = 2

    # --- Step 3: attempt conversion to float ---
    converted = pd.to_numeric(df_tyre_wear_final_replaced[col], errors="coerce")

    # --- Step 4: detect invalid values (exclude safe words already handled) ---
    mask_invalid = (
        df_tyre_wear_final_replaced[col].notna()
        & converted.isna()
    )

    if mask_invalid.any():
        temp = df_tyre_wear_final_replaced.loc[mask_invalid, ["filename"]].copy()
        temp["column"] = col
        temp["invalid_value"] = df_tyre_wear_final_replaced.loc[mask_invalid, col].values
        invalid_values.append(temp)

    # --- Step 5: assign valid numeric values ---
    mask_valid = converted.notna()
    df_tyre_wear_final_replaced.loc[mask_valid, col] = converted[mask_valid]

# --- Step 6: report ---
if invalid_values:
    df_invalid_values = pd.concat(invalid_values, ignore_index=True)
    df_invalid_values = df_invalid_values.sort_values("filename").reset_index(drop=True)

    print("❌ Values NOT converted to float:")
    # display(df_invalid_values)
else:
    print("✅ All groove values successfully converted to float.")


❌ Values NOT converted to float:


### Airbag Pressure

In [244]:
df_airbag_pressure = dfs_rsd.get("airbag_pressure")

for df in [df_airbag_pressure]:
    if df is not None:
        df.columns = df.columns.str.strip()

df_airbag_pressure = df_airbag_pressure.add_prefix("airbag_pressure.")

df_airbag_pressure.rename(columns={
    "airbag_pressure.workorder_id": "workorder_id",
    "airbag_pressure.filename": "filename",
    "airbag_pressure.approval.date": "airbag_pressure.approval_date",
    "airbag_pressure.approval.technician_id": "airbag_pressure.technician_id",
    "airbag_pressure.approval.supervisor_id": "airbag_pressure.supervisor_id",
}, inplace=True)


In [245]:
# Ensure required columns exist
required_cols = ["workorder_id", "functional_location", "work_request", "target_date"]
df_notification_subset = df_notification[required_cols].drop_duplicates()

# Merge functional_location into tyre_wear
df_airbag_pressure_final = df_airbag_pressure.merge(
    df_notification_subset,
    on="workorder_id",
    how="left"
)

# Optional: reorder columns (workorder_id, filename first)
cols = df_airbag_pressure_final.columns.tolist()

preferred_order = ["filename", "workorder_id", "functional_location", "work_request", "target_date"]
remaining_cols = [c for c in cols if c not in preferred_order]

df_airbag_pressure_final = df_airbag_pressure_final[
    preferred_order + remaining_cols
]

df_airbag_pressure_final.head()

,filename,workorder_id,functional_location,work_request,target_date,airbag_pressure.bogie1.bogie_sn,airbag_pressure.bogie1.pressure_before,airbag_pressure.bogie1.pressure_after,airbag_pressure.bogie1.height_before,airbag_pressure.bogie1.height_after,...,airbag_pressure.bogie7.height_before,airbag_pressure.bogie7.height_after,airbag_pressure.bogie8.bogie_sn,airbag_pressure.bogie8.pressure_before,airbag_pressure.bogie8.pressure_after,airbag_pressure.bogie8.height_before,airbag_pressure.bogie8.height_after,airbag_pressure.approval_date,airbag_pressure.technician_id,airbag_pressure.supervisor_id
0,RS_PM_WEK_4000586856.pdf,4000586856,RSV029,WEK3,24/02/2024,,5,5,241,241,...,242,242,,5,5,241,241,25/02/2024,11515,7127
1,RS_PM_MTH_4000464193.pdf,4000464193,RSV025,MTH,12/05/2022,231,5.1,,243,,...,241,,219,5.1,,240,,12/05/2022,7306,7066
2,RS_PM_MTH_4000446287.pdf,4000446287,RSV027,MTH,25/01/2022,257,4.8,4.8,243,243,...,244,244,264,4.8,4.8,242,242,25/01/2022,7205,7127
3,RS_PM_WEK_4000558454.pdf,4000558454,RSV022,WEK3,07/10/2023,244,5.0,,241,,...,241,,246,5,,241,,09/10/2023,7205,7192
4,RS_PM_MTH_4000464732.pdf,4000464732,RSV027,MTH,17/05/2022,257,4.9,,241,,...,241,,264,4.9,,241,,17/05/2022,7205,7127


In [246]:
before_cols = [c for c in df_airbag_pressure_final.columns if c.endswith("_before")]
after_cols  = [c.replace("_before", "_after") for c in before_cols
               if c.replace("_before", "_after") in df_airbag_pressure_final.columns]
bogiesn_cols = [c for c in df_airbag_pressure_final.columns if c.endswith("bogie_sn")]

cols_to_check = before_cols + after_cols + bogiesn_cols

print(cols_to_check)

invalid_records = []

for col in cols_to_check:
    series = df_airbag_pressure_final[col]

    # normalize text for checking
    normalized = series.astype(str).str.strip().str.upper()

    # treat 'NEW' as valid (exclude from invalid check)
    # is_new = normalized == "NEW"

    # numeric conversion
    numeric = pd.to_numeric(series, errors="coerce")
    df_airbag_pressure_final[col] = pd.to_numeric(df_airbag_pressure_final[col], errors="coerce")

    # invalid = not null, not NEW, but cannot convert to number
    mask_invalid = (
        series.notna() &
        (~is_new) &
        numeric.isna()
    )

    if mask_invalid.any():
        temp = df_airbag_pressure_final.loc[mask_invalid, ["filename", col]].copy()
        temp["column"] = col
        temp.rename(columns={col: "invalid_value"}, inplace=True)
        invalid_records.append(temp)

invalid_df = pd.concat(invalid_records, ignore_index=True) if invalid_records else pd.DataFrame()

# invalid_df


['airbag_pressure.bogie1.pressure_before', 'airbag_pressure.bogie1.height_before', 'airbag_pressure.bogie2.pressure_before', 'airbag_pressure.bogie2.height_before', 'airbag_pressure.bogie3.pressure_before', 'airbag_pressure.bogie3.height_before', 'airbag_pressure.bogie4.pressure_before', 'airbag_pressure.bogie4.height_before', 'airbag_pressure.bogie5.pressure_before', 'airbag_pressure.bogie5.height_before', 'airbag_pressure.bogie6.pressure_before', 'airbag_pressure.bogie6.height_before', 'airbag_pressure.bogie7.pressure_before', 'airbag_pressure.bogie7.height_before', 'airbag_pressure.bogie8.pressure_before', 'airbag_pressure.bogie8.height_before', 'airbag_pressure.bogie1.pressure_after', 'airbag_pressure.bogie1.height_after', 'airbag_pressure.bogie2.pressure_after', 'airbag_pressure.bogie2.height_after', 'airbag_pressure.bogie3.pressure_after', 'airbag_pressure.bogie3.height_after', 'airbag_pressure.bogie4.pressure_after', 'airbag_pressure.bogie4.height_after', 'airbag_pressure.bogie5

In [247]:
import re

df_bogie_sn_airbag = df_bogie_sn.copy()

car_order = ["eca1", "ica2", "ica3", "eca4"]
car_to_start_bogie = {
    "eca1": 1,
    "ica2": 3,
    "ica3": 5,
    "eca4": 7,
}

rename_map = {}

for col in df_bogie_sn_airbag.columns:
    if not col.endswith("bogie_sn"):
        continue

    # match: tyre_pressure.eca1.bogie1.bogie_sn
    m = re.search(r"(eca1|ica2|ica3|eca4)\.bogie([12])\.bogie_sn$", col)
    if not m:
        continue

    car, bogie = m.groups()
    bogie_idx = car_to_start_bogie[car] + (int(bogie) - 1)

    rename_map[col] = f"airbag_pressure.bogie{bogie_idx}.bogie_sn"

df_bogie_sn_airbag.rename(columns=rename_map, inplace=True)

bogie_cols_airbag = [
    c for c in df_bogie_sn_airbag.columns
    if c != "filename" and c in df_airbag_pressure_final.columns
]

df_airbag_pressure_final_replaced = df_airbag_pressure_final.copy()

df_airbag_pressure_final_replaced.set_index("filename", inplace=True)
df_bogie_sn_airbag.set_index("filename", inplace=True)

common_filenames = df_airbag_pressure_final_replaced.index.intersection(
    df_bogie_sn_airbag.index
)

df_airbag_pressure_final_replaced.loc[
    common_filenames, bogie_cols_airbag
] = df_bogie_sn_airbag.loc[
    common_filenames, bogie_cols_airbag
]

df_airbag_pressure_final_replaced.reset_index(inplace=True)
df_bogie_sn_airbag.reset_index(inplace=True)

df_airbag_pressure_final_replaced.head()

C:\Users\win 11\AppData\Local\Temp\ipykernel_10000\1805129746.py:45: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '<IntegerArray>
[273, 231, 257, 244, 257, 246, 257, 251, 224, 254,
 ...
 251, 246, 252, 208, 268, 207, 267, 244, 219, 253]
Length: 1205, dtype: Int64' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_airbag_pressure_final_replaced.loc[
C:\Users\win 11\AppData\Local\Temp\ipykernel_10000\1805129746.py:45: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '<IntegerArray>
[282, 247, 258, 228, 258, 205, 258, 205, 203, 205,
 ...
 205, 201, 219, 271, 271, 230, 262, 228, 282, 222]
Length: 1205, dtype: Int64' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_airbag_pressure_final_replaced.loc[
C:\Users\win 11\AppData\Local\Temp\ipykernel

,filename,workorder_id,functional_location,work_request,target_date,airbag_pressure.bogie1.bogie_sn,airbag_pressure.bogie1.pressure_before,airbag_pressure.bogie1.pressure_after,airbag_pressure.bogie1.height_before,airbag_pressure.bogie1.height_after,...,airbag_pressure.bogie7.height_before,airbag_pressure.bogie7.height_after,airbag_pressure.bogie8.bogie_sn,airbag_pressure.bogie8.pressure_before,airbag_pressure.bogie8.pressure_after,airbag_pressure.bogie8.height_before,airbag_pressure.bogie8.height_after,airbag_pressure.approval_date,airbag_pressure.technician_id,airbag_pressure.supervisor_id
0,RS_PM_WEK_4000586856.pdf,4000586856,RSV029,WEK3,24/02/2024,273.0,5.0,5.0,241.0,241.0,...,242.0,242.0,226.0,5.0,5.0,241.0,241.0,25/02/2024,11515,7127
1,RS_PM_MTH_4000464193.pdf,4000464193,RSV025,MTH,12/05/2022,231.0,5.1,NaN,243.0,NaN,...,241.0,NaN,214.0,5.1,NaN,240.0,NaN,12/05/2022,7306,7066
2,RS_PM_MTH_4000446287.pdf,4000446287,RSV027,MTH,25/01/2022,257.0,4.8,4.8,243.0,243.0,...,244.0,244.0,264.0,4.8,4.8,242.0,242.0,25/01/2022,7205,7127
3,RS_PM_WEK_4000558454.pdf,4000558454,RSV022,WEK3,07/10/2023,244.0,5.0,NaN,241.0,NaN,...,241.0,NaN,246.0,5.0,NaN,241.0,NaN,09/10/2023,7205,7192
4,RS_PM_MTH_4000464732.pdf,4000464732,RSV027,MTH,17/05/2022,257.0,4.9,NaN,241.0,NaN,...,241.0,NaN,264.0,4.9,NaN,241.0,NaN,17/05/2022,7205,7127


In [248]:
import re

pattern = re.compile(r"^(.*)\.(pressure|height)_before$")

for col in df_airbag_pressure_final_replaced.columns:
    m = pattern.match(col)
    if not m:
        continue

    base, category = m.groups()
    before_col = col
    after_col = f"{base}.{category}_after"

    if after_col not in df_airbag_pressure_final_replaced.columns:
        continue

    before_series = df_airbag_pressure_final_replaced[before_col]
    after_series  = df_airbag_pressure_final_replaced[after_col]

    # before → after
    mask_after_missing = (
        before_series.notna() &
        (
            after_series.isna() |
            (after_series.astype(str).str.strip() == "")
        )
    )

    df_airbag_pressure_final_replaced.loc[mask_after_missing, after_col] = before_series[mask_after_missing]

    # after → before
    mask_before_missing = (
        after_series.notna() &
        (
            before_series.isna() |
            (before_series.astype(str).str.strip() == "")
        )
    )

    df_airbag_pressure_final_replaced.loc[mask_before_missing, before_col] = after_series[mask_before_missing]

df_airbag_pressure_final_replaced.head()


,filename,workorder_id,functional_location,work_request,target_date,airbag_pressure.bogie1.bogie_sn,airbag_pressure.bogie1.pressure_before,airbag_pressure.bogie1.pressure_after,airbag_pressure.bogie1.height_before,airbag_pressure.bogie1.height_after,...,airbag_pressure.bogie7.height_before,airbag_pressure.bogie7.height_after,airbag_pressure.bogie8.bogie_sn,airbag_pressure.bogie8.pressure_before,airbag_pressure.bogie8.pressure_after,airbag_pressure.bogie8.height_before,airbag_pressure.bogie8.height_after,airbag_pressure.approval_date,airbag_pressure.technician_id,airbag_pressure.supervisor_id
0,RS_PM_WEK_4000586856.pdf,4000586856,RSV029,WEK3,24/02/2024,273.0,5.0,5.0,241.0,241.0,...,242.0,242.0,226.0,5.0,5.0,241.0,241.0,25/02/2024,11515,7127
1,RS_PM_MTH_4000464193.pdf,4000464193,RSV025,MTH,12/05/2022,231.0,5.1,5.1,243.0,243.0,...,241.0,241.0,214.0,5.1,5.1,240.0,240.0,12/05/2022,7306,7066
2,RS_PM_MTH_4000446287.pdf,4000446287,RSV027,MTH,25/01/2022,257.0,4.8,4.8,243.0,243.0,...,244.0,244.0,264.0,4.8,4.8,242.0,242.0,25/01/2022,7205,7127
3,RS_PM_WEK_4000558454.pdf,4000558454,RSV022,WEK3,07/10/2023,244.0,5.0,5.0,241.0,241.0,...,241.0,241.0,246.0,5.0,5.0,241.0,241.0,09/10/2023,7205,7192
4,RS_PM_MTH_4000464732.pdf,4000464732,RSV027,MTH,17/05/2022,257.0,4.9,4.9,241.0,241.0,...,241.0,241.0,264.0,4.9,4.9,241.0,241.0,17/05/2022,7205,7127


In [249]:
# combine all .before columns in one list and find unique values
before_cols = [
    c for c in df_airbag_pressure_final_replaced.columns
    if c.endswith("_before")
]
unique_before_values = set()
for col in before_cols:
    unique_before_values.update(df_airbag_pressure_final_replaced[col].dropna().unique())
print(unique_before_values)

{np.float64(0.0), np.float64(1.8), np.float64(1.5), np.float64(3.0), np.float64(4.8), np.float64(5.8), np.float64(5.1), np.float64(5.0), np.float64(4.9), np.float64(4.5), np.float64(5.2), np.float64(4.3), np.float64(5.5), np.float64(5.3), np.float64(6.0), np.float64(9.8), np.float64(7.0), np.float64(8.0), np.float64(9.9), np.float64(3.5), np.float64(2.5), np.float64(4.0), np.float64(23.0), np.float64(24.5), np.float64(24.0), np.float64(24.3), np.float64(29.0), np.float64(6.5), np.float64(0.5), np.float64(7.5), np.float64(40.0), np.float64(41.0), np.float64(8.5), np.float64(44.0), np.float64(45.0), np.float64(9.5), np.float64(47.0), np.float64(48.0), np.float64(46.0), np.float64(50.0), np.float64(51.0), np.float64(52.0), np.float64(3.9), np.float64(138.0), np.float64(139.0), np.float64(140.0), np.float64(143.0), np.float64(144.0), np.float64(4.4), np.float64(4.51), np.float64(5.4), np.float64(5.9), np.float64(200.0), np.float64(201.0), np.float64(3.8), np.float64(210.0), np.float64(754.

### CCEB

In [250]:
df_cceb = dfs_rsd.get("cceb")

for df in [df_cceb]:
    if df is not None:
        df.columns = df.columns.str.strip()

df_cceb = df_cceb.add_prefix("cceb.")

df_cceb.rename(columns={
    "cceb.workorder_id": "workorder_id",
    "cceb.filename": "filename",
    "cceb.approval.date": "cceb.approval_date",
    "cceb.approval.technician_id": "cceb.technician_id",
    "cceb.approval.supervisor_id": "cceb.supervisor_id",
}, inplace=True)

df_cceb.rename(
    columns=lambda c: (
        c.replace("cceb.currect_collector.", "cceb.current_collector.")
        if c.startswith("cceb.currect_collector.")
        else c
    ),
    inplace=True
)

df_cceb.head()

,cceb.train_no,cceb.location,cceb.frequency,cceb.date,cceb.current_collector.ca,cceb.current_collector.cb,cceb.current_collector.cc,cceb.current_collector.cd,cceb.current_collector.ce,cceb.current_collector.cf,...,cceb.earthing.e2,cceb.earthing.e3,cceb.earthing.e4,cceb.technician_detail.technician_id,cceb.technician_detail.technician_date,cceb.supervisor_detail.supervisor_id,cceb.supervisor_detail.supervisor_date,cceb.remarks,workorder_id,filename
0,29,BRICKFIELDS DEPOT,WEEKLY,,7,13,12,12,13,13,...,N/A,12,N/A,11515,25/02/2024,7127,25/02/2024,,4000586856,RS_PM_WEK_4000586856.pdf
1,25,BRICKFIELDS DEPOT,MONTHLY,12/07/2022,NEW,7,10,9,7,NEW,...,7,8,6,7276,12/05/2022,7066,12/05/2022,,4000464193,RS_PM_MTH_4000464193.pdf
2,27,BRICKFIELDS DEPOT,MONTHLY,25/01/2022,8.6,10.5,9.5,11.0,9.5,11.0,...,7.0,10.0,9.5,7205,25/01/2022,7127,25/01/2022,,4000446287,RS_PM_MTH_4000446287.pdf
3,22,BRICKFIELDS DEPOT,WEEKLY,09/10/2023,12.5,12.0,11.5,10.5,10.5,11.5,...,9.5,5.5,NA,7205,09/10/2023,7192,09/10/2023,NA,4000558454,RS_PM_WEK_4000558454.pdf
4,27,BRICKFIELDS DEPOT,MONTHLY,17/05/2022,4.0,CHANGE NEW,5.5,10.5,CHANGE NEW,CHANGE NEW,...,11.5,12.5,NA,7205,17/05/2022,7127,17/05/2022,,4000464732,RS_PM_MTH_4000464732.pdf


In [251]:
# Ensure required columns exist
required_cols = ["workorder_id", "functional_location", "work_request", "target_date"]
df_notification_subset = df_notification[required_cols].drop_duplicates()

# Merge functional_location into tyre_wear
df_cceb_final = df_cceb.merge(
    df_notification_subset,
    on="workorder_id",
    how="left"
)

# Optional: reorder columns (workorder_id, filename first)
cols = df_cceb_final.columns.tolist()

preferred_order = ["filename", "workorder_id", "functional_location", "work_request", "target_date"]
remaining_cols = [c for c in cols if c not in preferred_order]

df_cceb_final = df_cceb_final[
    preferred_order + remaining_cols
]

df_cceb_final.head()

,filename,workorder_id,functional_location,work_request,target_date,cceb.train_no,cceb.location,cceb.frequency,cceb.date,cceb.current_collector.ca,...,cceb.current_collector.cf,cceb.earthing.e1,cceb.earthing.e2,cceb.earthing.e3,cceb.earthing.e4,cceb.technician_detail.technician_id,cceb.technician_detail.technician_date,cceb.supervisor_detail.supervisor_id,cceb.supervisor_detail.supervisor_date,cceb.remarks
0,RS_PM_WEK_4000586856.pdf,4000586856,RSV029,WEK3,24/02/2024,29,BRICKFIELDS DEPOT,WEEKLY,,7,...,13,N/A,N/A,12,N/A,11515,25/02/2024,7127,25/02/2024,
1,RS_PM_MTH_4000464193.pdf,4000464193,RSV025,MTH,12/05/2022,25,BRICKFIELDS DEPOT,MONTHLY,12/07/2022,NEW,...,NEW,NA,7,8,6,7276,12/05/2022,7066,12/05/2022,
2,RS_PM_MTH_4000446287.pdf,4000446287,RSV027,MTH,25/01/2022,27,BRICKFIELDS DEPOT,MONTHLY,25/01/2022,8.6,...,11.0,7.0,7.0,10.0,9.5,7205,25/01/2022,7127,25/01/2022,
3,RS_PM_WEK_4000558454.pdf,4000558454,RSV022,WEK3,07/10/2023,22,BRICKFIELDS DEPOT,WEEKLY,09/10/2023,12.5,...,11.5,NA,9.5,5.5,NA,7205,09/10/2023,7192,09/10/2023,NA
4,RS_PM_MTH_4000464732.pdf,4000464732,RSV027,MTH,17/05/2022,27,BRICKFIELDS DEPOT,MONTHLY,17/05/2022,4.0,...,CHANGE NEW,NA,11.5,12.5,NA,7205,17/05/2022,7127,17/05/2022,


In [252]:
# words to EXCLUDE from invalid checking (case-insensitive)
exclude_words = [
    "NEW",
    "REPLACED",
    "REPLACE",
    "X", "-", "OK",
    "CHANGE", "NO STOCK", "NEW SET",
    "NO SHOE", "PART", "NIL", "N/F", "NA", "N/E",
    "OUT OF SPEC", "USED", "N/A"
]

current_collector_cols = [
    c for c in df_cceb_final.columns
    if c.startswith("cceb.current_collector")
]

earthing_cols = [
    c for c in df_cceb_final.columns
    if c.startswith("cceb.earthing")
]

cols_to_check = current_collector_cols + earthing_cols
# cols_to_check
invalid_records = []

for col in cols_to_check:
    series = df_cceb_final[col]

    # normalize text
    normalized = series.astype(str).str.strip().str.upper()

    # check if value contains ANY excluded word
    contains_excluded_word = normalized.apply(
        lambda x: any(word in x for word in exclude_words)
    )

    # numeric conversion
    numeric = pd.to_numeric(series, errors="coerce")

    # invalid condition
    mask_invalid = (
        series.notna() &
        (~contains_excluded_word) &
        numeric.isna()
    )

    if mask_invalid.any():
        temp = df_cceb_final.loc[mask_invalid, ["filename", col]].copy()
        temp["column"] = col
        temp.rename(columns={col: "invalid_value"}, inplace=True)
        invalid_records.append(temp)

if invalid_records:
    invalid_df = pd.concat(invalid_records, ignore_index=True)
    invalid_df.sort_values(["filename", "column"], inplace=True)
else:
    # create empty DataFrame with expected columns
    invalid_df = pd.DataFrame(columns=["filename", "invalid_value", "column"])

# invalid_df


In [253]:
df_cceb_final.head()

,filename,workorder_id,functional_location,work_request,target_date,cceb.train_no,cceb.location,cceb.frequency,cceb.date,cceb.current_collector.ca,...,cceb.current_collector.cf,cceb.earthing.e1,cceb.earthing.e2,cceb.earthing.e3,cceb.earthing.e4,cceb.technician_detail.technician_id,cceb.technician_detail.technician_date,cceb.supervisor_detail.supervisor_id,cceb.supervisor_detail.supervisor_date,cceb.remarks
0,RS_PM_WEK_4000586856.pdf,4000586856,RSV029,WEK3,24/02/2024,29,BRICKFIELDS DEPOT,WEEKLY,,7,...,13,N/A,N/A,12,N/A,11515,25/02/2024,7127,25/02/2024,
1,RS_PM_MTH_4000464193.pdf,4000464193,RSV025,MTH,12/05/2022,25,BRICKFIELDS DEPOT,MONTHLY,12/07/2022,NEW,...,NEW,NA,7,8,6,7276,12/05/2022,7066,12/05/2022,
2,RS_PM_MTH_4000446287.pdf,4000446287,RSV027,MTH,25/01/2022,27,BRICKFIELDS DEPOT,MONTHLY,25/01/2022,8.6,...,11.0,7.0,7.0,10.0,9.5,7205,25/01/2022,7127,25/01/2022,
3,RS_PM_WEK_4000558454.pdf,4000558454,RSV022,WEK3,07/10/2023,22,BRICKFIELDS DEPOT,WEEKLY,09/10/2023,12.5,...,11.5,NA,9.5,5.5,NA,7205,09/10/2023,7192,09/10/2023,NA
4,RS_PM_MTH_4000464732.pdf,4000464732,RSV027,MTH,17/05/2022,27,BRICKFIELDS DEPOT,MONTHLY,17/05/2022,4.0,...,CHANGE NEW,NA,11.5,12.5,NA,7205,17/05/2022,7127,17/05/2022,


In [254]:
def normalize(text):
    return (
        str(text)
        .strip()
        .upper()
        .replace(".", "")
        .replace("-", "")
        .replace("/", "")
        .replace(" ", "")
    )

exclude_set = {normalize(w) for w in exclude_words if w}

def convert_to_float_refined(val):
    # Handle actual nulls or empty cells first
    if pd.isna(val) or str(val).strip() == "":
        return np.nan
    
    val_str = str(val).strip()
    val_norm = normalize(val_str)
    
    # 2. Use EXACT match (in exclude_set) instead of substring match (any ... in ...)
    if val_norm in exclude_set:
        return val_str  # Keep original string
    
    # 3. Try to convert to float
    try:
        return float(val_str)
    except (ValueError, TypeError):
        return val_str  # Keep as string if it's not a number and not in exclude list

# Apply the conversion
for col in cols_to_check:
    df_cceb_final[col] = df_cceb_final[col].apply(convert_to_float_refined)

# 4. VERIFY the types of individual elements
print(f"Column Dtype: {df_cceb_final[cols_to_check[0]].dtype}")
print(f"Type of first element: {type(df_cceb_final[cols_to_check[0]].iloc[0])}")


df_cceb_final[cols_to_check].head()

Column Dtype: object
Type of first element: <class 'float'>


,cceb.current_collector.ca,cceb.current_collector.cb,cceb.current_collector.cc,cceb.current_collector.cd,cceb.current_collector.ce,cceb.current_collector.cf,cceb.earthing.e1,cceb.earthing.e2,cceb.earthing.e3,cceb.earthing.e4
0,7.0,13.0,12.0,12.0,13.0,13.0,N/A,N/A,12.0,N/A
1,NEW,7.0,10.0,9.0,7.0,NEW,NA,7.0,8.0,6.0
2,8.6,10.5,9.5,11.0,9.5,11.0,7.0,7.0,10.0,9.5
3,12.5,12.0,11.5,10.5,10.5,11.5,NA,9.5,5.5,NA
4,4.0,CHANGE NEW,5.5,10.5,CHANGE NEW,CHANGE NEW,NA,11.5,12.5,NA


In [255]:
df_cceb_final.head()

,filename,workorder_id,functional_location,work_request,target_date,cceb.train_no,cceb.location,cceb.frequency,cceb.date,cceb.current_collector.ca,...,cceb.current_collector.cf,cceb.earthing.e1,cceb.earthing.e2,cceb.earthing.e3,cceb.earthing.e4,cceb.technician_detail.technician_id,cceb.technician_detail.technician_date,cceb.supervisor_detail.supervisor_id,cceb.supervisor_detail.supervisor_date,cceb.remarks
0,RS_PM_WEK_4000586856.pdf,4000586856,RSV029,WEK3,24/02/2024,29,BRICKFIELDS DEPOT,WEEKLY,,7.0,...,13.0,N/A,N/A,12.0,N/A,11515,25/02/2024,7127,25/02/2024,
1,RS_PM_MTH_4000464193.pdf,4000464193,RSV025,MTH,12/05/2022,25,BRICKFIELDS DEPOT,MONTHLY,12/07/2022,NEW,...,NEW,NA,7.0,8.0,6.0,7276,12/05/2022,7066,12/05/2022,
2,RS_PM_MTH_4000446287.pdf,4000446287,RSV027,MTH,25/01/2022,27,BRICKFIELDS DEPOT,MONTHLY,25/01/2022,8.6,...,11.0,7.0,7.0,10.0,9.5,7205,25/01/2022,7127,25/01/2022,
3,RS_PM_WEK_4000558454.pdf,4000558454,RSV022,WEK3,07/10/2023,22,BRICKFIELDS DEPOT,WEEKLY,09/10/2023,12.5,...,11.5,NA,9.5,5.5,NA,7205,09/10/2023,7192,09/10/2023,NA
4,RS_PM_MTH_4000464732.pdf,4000464732,RSV027,MTH,17/05/2022,27,BRICKFIELDS DEPOT,MONTHLY,17/05/2022,4.0,...,CHANGE NEW,NA,11.5,12.5,NA,7205,17/05/2022,7127,17/05/2022,


### Air Standup Test

In [256]:
df_air_standup = dfs_rsd.get("air_standup")

for df in [df_air_standup]:
    if df is not None:
        df.columns = df.columns.str.strip()

df_air_standup = df_air_standup.add_prefix("air_standup.")

df_air_standup.rename(columns={
    "air_standup.workorder_id": "workorder_id",
    "air_standup.filename": "filename",
    "air_standup.date": "air_standup.approval_date",
    "air_standup.approval.technician_id": "air_standup.technician_id",
    "air_standup.approval.supervisor_id": "air_standup.supervisor_id",
}, inplace=True)

cols = ["air_standup.after_5_minutes", "air_standup.after_10_minutes"]

df_air_standup[cols] = df_air_standup[cols].replace(r"^\s*$", np.nan, regex=True)
df_air_standup[cols] = df_air_standup[cols].apply(pd.to_numeric, errors="coerce")
df_air_standup = df_air_standup.dropna(subset=cols)

df_air_standup["air_standup.pressure_difference"] = (
    df_air_standup["air_standup.after_10_minutes"] - df_air_standup["air_standup.after_5_minutes"]
).abs().round(2)

df_air_standup.head()

,air_standup.after_5_minutes,air_standup.after_10_minutes,air_standup.approval_date,air_standup.technician_id,air_standup.supervisor_id,workorder_id,filename,air_standup.pressure_difference
49,8.4,7.3,09/06/2022,7127,7127,4000469186,RS_PM_YRL_4000469186.pdf,1.1
757,9.0,8.3,11/05/2023,7127,7127,4000529588,RS_PM_YRL_4000529588.pdf,0.7
758,3.5,4.9,09/03/2022,11515,6196,4000457606,RS_PM_YRL_4000457606.pdf,1.4
760,9.2,8.8,03/04/2023,"6196, 10016745",7196,4000523784,RS_PM_YRL_4000523784.pdf,0.4
764,8.8,8.5,13/05/2021,7205,7232,4000462732,RS_PM_YRL_4000462732.pdf,0.3


In [257]:
required_cols = ["workorder_id", "functional_location", "work_request", "target_date"]
df_notification_subset = df_notification[required_cols].drop_duplicates()

df_air_standup_final = df_air_standup.merge(
    df_notification_subset,
    on="workorder_id",
    how="left"
)

cols = df_air_standup_final.columns.tolist()

preferred_order = ["filename", "workorder_id", "functional_location", "work_request", "target_date"]
remaining_cols = ["air_standup.after_5_minutes", "air_standup.after_10_minutes", "air_standup.pressure_difference", "air_standup.approval_date", "air_standup.technician_id", "air_standup.supervisor_id"]

df_air_standup_final = df_air_standup_final[
    preferred_order + remaining_cols
]

df_air_standup_final.head()

,filename,workorder_id,functional_location,work_request,target_date,air_standup.after_5_minutes,air_standup.after_10_minutes,air_standup.pressure_difference,air_standup.approval_date,air_standup.technician_id,air_standup.supervisor_id
0,RS_PM_YRL_4000469186.pdf,4000469186,RSV025,YRL,09/06/2022,8.4,7.3,1.1,09/06/2022,7127,7127
1,RS_PM_YRL_4000529588.pdf,4000529588,RSV025,YRL,11/05/2023,9.0,8.3,0.7,11/05/2023,7127,7127
2,RS_PM_YRL_4000457606.pdf,4000457606,RSV021,YRL,01/04/2022,3.5,4.9,1.4,09/03/2022,11515,6196
3,RS_PM_YRL_4000523784.pdf,4000523784,RSV022,YRL,01/04/2023,9.2,8.8,0.4,03/04/2023,"6196, 10016745",7196
4,RS_PM_YRL_4000462732.pdf,4000462732,RSV023,YRL,01/05/2022,8.8,8.5,0.3,13/05/2021,7205,7232


### Water Ponding

In [258]:
df_waterponding = dfs_rsd.get("water_ponding")

for df in [df_waterponding]:
    if df is not None:
        df.columns = df.columns.str.strip()

df_waterponding = df_waterponding.add_prefix("water_ponding.")

df_waterponding.rename(columns={
    "water_ponding.workorder_id": "workorder_id",
    "water_ponding.filename": "filename",
    "water_ponding.date": "water_ponding.approval_date",
    "water_ponding.approval.technician_id": "water_ponding.technician_id",
    "water_ponding.approval.supervisor_id": "water_ponding.supervisor_id",
}, inplace=True)

df_waterponding.head()

,water_ponding.aps_auxiliary_power_supply.eca1,water_ponding.aps_auxiliary_power_supply.ica2,water_ponding.aps_auxiliary_power_supply.ica3,water_ponding.aps_auxiliary_power_supply.eca4,water_ponding.mcal_main_cabinet_apron_left.eca1,water_ponding.mcal_main_cabinet_apron_left.ica2,water_ponding.mcal_main_cabinet_apron_left.ica3,water_ponding.mcal_main_cabinet_apron_left.eca4,water_ponding.mcar_main_cabinet_apron_right.eca1,water_ponding.mcar_main_cabinet_apron_right.ica2,...,water_ponding.traction_inverter_2.ica2,water_ponding.traction_inverter_4.ica3,water_ponding.traction_inverter_5.ica3,water_ponding.traction_inverter_6.eca4,water_ponding.approval_date,water_ponding.technician_id,water_ponding.supervisor_id,water_ponding.traction_inverter_3.ica2,workorder_id,filename
0,True,True,True,True,True,True,True,True,True,True,...,True,True,True,True,25/02/2024,"19580,20674",7127,True,4000586856,RS_PM_WEK_4000586856.pdf
1,True,True,True,True,True,True,True,True,True,True,...,True,True,True,True,12/05/2022,7306,7066,True,4000464193,RS_PM_MTH_4000464193.pdf
2,True,True,True,True,True,True,True,True,True,True,...,True,True,True,True,25/01/2022,11139,7127,True,4000446287,RS_PM_MTH_4000446287.pdf
3,True,True,True,True,True,True,True,True,True,True,...,True,True,True,True,09/10/2023,,,True,4000558454,RS_PM_WEK_4000558454.pdf
4,True,True,True,True,True,True,True,True,True,True,...,True,True,True,True,17/05/2022,9031,7127,True,4000464732,RS_PM_MTH_4000464732.pdf


In [259]:
required_cols = ["workorder_id", "functional_location", "work_request", "target_date"]
df_notification_subset = df_notification[required_cols].drop_duplicates()

df_waterponding_final = df_waterponding.merge(
    df_notification_subset,
    on="workorder_id",
    how="left"
)

cols = df_waterponding_final.columns.tolist()

eca1_cols = [c for c in cols if c.endswith(".eca1")]
ica2_cols = [c for c in cols if c.endswith(".ica2")]
ica3_cols = [c for c in cols if c.endswith(".ica3")]
eca4_cols = [c for c in cols if c.endswith(".eca4")]

waterponding_val_cols = eca1_cols + ica2_cols + ica3_cols + eca4_cols

preferred_order = ["filename", "workorder_id", "functional_location", "work_request", "target_date"]
remaining_cols = ["water_ponding.approval_date", "water_ponding.technician_id", "water_ponding.supervisor_id"]

df_waterponding_final = df_waterponding_final[
    preferred_order + waterponding_val_cols + remaining_cols
]

df_waterponding_final.head()

,filename,workorder_id,functional_location,work_request,target_date,water_ponding.aps_auxiliary_power_supply.eca1,water_ponding.mcal_main_cabinet_apron_left.eca1,water_ponding.mcar_main_cabinet_apron_right.eca1,water_ponding.traction_inverter_1.eca1,water_ponding.aps_auxiliary_power_supply.ica2,...,water_ponding.mcar_main_cabinet_apron_right.ica3,water_ponding.traction_inverter_4.ica3,water_ponding.traction_inverter_5.ica3,water_ponding.aps_auxiliary_power_supply.eca4,water_ponding.mcal_main_cabinet_apron_left.eca4,water_ponding.mcar_main_cabinet_apron_right.eca4,water_ponding.traction_inverter_6.eca4,water_ponding.approval_date,water_ponding.technician_id,water_ponding.supervisor_id
0,RS_PM_WEK_4000586856.pdf,4000586856,RSV029,WEK3,24/02/2024,True,True,True,True,True,...,True,True,True,True,True,True,True,25/02/2024,"19580,20674",7127
1,RS_PM_MTH_4000464193.pdf,4000464193,RSV025,MTH,12/05/2022,True,True,True,True,True,...,True,True,True,True,True,True,True,12/05/2022,7306,7066
2,RS_PM_MTH_4000446287.pdf,4000446287,RSV027,MTH,25/01/2022,True,True,True,True,True,...,True,True,True,True,True,True,True,25/01/2022,11139,7127
3,RS_PM_WEK_4000558454.pdf,4000558454,RSV022,WEK3,07/10/2023,True,True,True,True,True,...,True,True,True,True,True,True,True,09/10/2023,,
4,RS_PM_MTH_4000464732.pdf,4000464732,RSV027,MTH,17/05/2022,True,True,True,True,True,...,True,True,True,True,True,True,True,17/05/2022,9031,7127


### Cardan Shaft

In [260]:
df_cardanshaft = dfs_rsd.get("cardan_shaft")

for df in [df_cardanshaft]:
    if df is not None:
        df.columns = df.columns.str.strip()

df_cardanshaft = df_cardanshaft.add_prefix("cardan_shaft.")

df_cardanshaft.rename(columns={
    "cardan_shaft.workorder_id": "workorder_id",
    "cardan_shaft.filename": "filename",
    "cardan_shaft.date": "cardan_shaft.approval_date",
    "cardan_shaft.approval.technician_id": "cardan_shaft.technician_id",
    "cardan_shaft.approval.supervisor_id": "cardan_shaft.supervisor_id",
}, inplace=True)

df_cardanshaft.head()

,cardan_shaft.approval_date,cardan_shaft.technician_id,cardan_shaft.supervisor_id,cardan_shaft.bogie_checks.bogie2.long_cardan_shaft,cardan_shaft.bogie_checks.bogie2.short_cardan_shaft,cardan_shaft.bogie_checks.bogie3.long_cardan_shaft,cardan_shaft.bogie_checks.bogie3.short_cardan_shaft,cardan_shaft.bogie_checks.bogie4.long_cardan_shaft,cardan_shaft.bogie_checks.bogie4.short_cardan_shaft,cardan_shaft.bogie_checks.bogie5.long_cardan_shaft,cardan_shaft.bogie_checks.bogie5.short_cardan_shaft,cardan_shaft.bogie_checks.bogie6.long_cardan_shaft,cardan_shaft.bogie_checks.bogie6.short_cardan_shaft,cardan_shaft.bogie_checks.bogie7.long_cardan_shaft,cardan_shaft.bogie_checks.bogie7.short_cardan_shaft,workorder_id,filename
0,25/02/2024,"19580,20674",7127,True,True,True,True,True,True,True,True,True,True,True,True,4000586856,RS_PM_WEK_4000586856.pdf
1,12/05/2022,7232,7066,True,True,True,True,True,True,True,True,True,True,True,True,4000464193,RS_PM_MTH_4000464193.pdf
2,25/01/2022,11139,7127,True,True,True,True,True,True,True,True,True,True,True,True,4000446287,RS_PM_MTH_4000446287.pdf
3,09/10/2023,11083,7192,True,True,True,True,True,True,True,True,True,True,True,True,4000558454,RS_PM_WEK_4000558454.pdf
4,17/05/2022,MSF,7127,True,True,True,True,True,True,True,True,True,True,True,True,4000464732,RS_PM_MTH_4000464732.pdf


In [261]:
required_cols = ["workorder_id", "functional_location", "work_request", "target_date"]
df_notification_subset = df_notification[required_cols].drop_duplicates()

df_cardanshaft_final = df_cardanshaft.merge(
    df_notification_subset,
    on="workorder_id",
    how="left"
)

cols = df_cardanshaft_final.columns.tolist()

preferred_order = ["filename", "workorder_id", "functional_location", "work_request", "target_date"]
approval_cols = ["cardan_shaft.approval_date", "cardan_shaft.technician_id", "cardan_shaft.supervisor_id"]
remaining_cols = [c for c in cols if c not in preferred_order + approval_cols]

df_cardanshaft_final = df_cardanshaft_final[
    preferred_order + remaining_cols + approval_cols
]

df_cardanshaft_final.head()

,filename,workorder_id,functional_location,work_request,target_date,cardan_shaft.bogie_checks.bogie2.long_cardan_shaft,cardan_shaft.bogie_checks.bogie2.short_cardan_shaft,cardan_shaft.bogie_checks.bogie3.long_cardan_shaft,cardan_shaft.bogie_checks.bogie3.short_cardan_shaft,cardan_shaft.bogie_checks.bogie4.long_cardan_shaft,cardan_shaft.bogie_checks.bogie4.short_cardan_shaft,cardan_shaft.bogie_checks.bogie5.long_cardan_shaft,cardan_shaft.bogie_checks.bogie5.short_cardan_shaft,cardan_shaft.bogie_checks.bogie6.long_cardan_shaft,cardan_shaft.bogie_checks.bogie6.short_cardan_shaft,cardan_shaft.bogie_checks.bogie7.long_cardan_shaft,cardan_shaft.bogie_checks.bogie7.short_cardan_shaft,cardan_shaft.approval_date,cardan_shaft.technician_id,cardan_shaft.supervisor_id
0,RS_PM_WEK_4000586856.pdf,4000586856,RSV029,WEK3,24/02/2024,True,True,True,True,True,True,True,True,True,True,True,True,25/02/2024,"19580,20674",7127
1,RS_PM_MTH_4000464193.pdf,4000464193,RSV025,MTH,12/05/2022,True,True,True,True,True,True,True,True,True,True,True,True,12/05/2022,7232,7066
2,RS_PM_MTH_4000446287.pdf,4000446287,RSV027,MTH,25/01/2022,True,True,True,True,True,True,True,True,True,True,True,True,25/01/2022,11139,7127
3,RS_PM_WEK_4000558454.pdf,4000558454,RSV022,WEK3,07/10/2023,True,True,True,True,True,True,True,True,True,True,True,True,09/10/2023,11083,7192
4,RS_PM_MTH_4000464732.pdf,4000464732,RSV027,MTH,17/05/2022,True,True,True,True,True,True,True,True,True,True,True,True,17/05/2022,MSF,7127


### Greasing Cardan Shaft

In [262]:
df_greasing_cardanshaft = dfs_rsd.get("greasing_cardan_shaft")

for df in [df_greasing_cardanshaft]:
    if df is not None:
        df.columns = df.columns.str.strip()

df_greasing_cardanshaft = df_greasing_cardanshaft.add_prefix("greasing_cardan_shaft.")

df_greasing_cardanshaft.rename(columns={
    "greasing_cardan_shaft.workorder_id": "workorder_id",
    "greasing_cardan_shaft.filename": "filename",
    "greasing_cardan_shaft.approval.date": "greasing_cardan_shaft.approval_date",
    "greasing_cardan_shaft.approval.technician_id": "greasing_cardan_shaft.technician_id",
    "greasing_cardan_shaft.approval.supervisor_id": "greasing_cardan_shaft.supervisor_id",
}, inplace=True)

df_greasing_cardanshaft.head()

,greasing_cardan_shaft.bogie2.long_cardan_shaft.1,greasing_cardan_shaft.bogie2.long_cardan_shaft.2,greasing_cardan_shaft.bogie2.long_cardan_shaft.3,greasing_cardan_shaft.bogie2.short_cardan_shaft.4,greasing_cardan_shaft.bogie2.short_cardan_shaft.5,greasing_cardan_shaft.bogie2.short_cardan_shaft.6,greasing_cardan_shaft.bogie3.long_cardan_shaft.1,greasing_cardan_shaft.bogie3.long_cardan_shaft.2,greasing_cardan_shaft.bogie3.long_cardan_shaft.3,greasing_cardan_shaft.bogie3.short_cardan_shaft.4,...,greasing_cardan_shaft.bogie7.long_cardan_shaft.2,greasing_cardan_shaft.bogie7.long_cardan_shaft.3,greasing_cardan_shaft.bogie7.short_cardan_shaft.4,greasing_cardan_shaft.bogie7.short_cardan_shaft.5,greasing_cardan_shaft.bogie7.short_cardan_shaft.6,greasing_cardan_shaft.approval_date,greasing_cardan_shaft.technician_id,greasing_cardan_shaft.supervisor_id,workorder_id,filename
0,,,,,,,,,,,...,,,,,,,,,4000586856,RS_PM_WEK_4000586856.pdf
1,,,,,,,,,,,...,,,,,,,,,4000464193,RS_PM_MTH_4000464193.pdf
2,,,,,,,,,,,...,,,,,,,,,4000446287,RS_PM_MTH_4000446287.pdf
3,,,,,,,,,,,...,,,,,,,,,4000558454,RS_PM_WEK_4000558454.pdf
4,,,,,,,,,,,...,,,,,,,,,4000464732,RS_PM_MTH_4000464732.pdf


In [263]:
required_cols = ["workorder_id", "functional_location", "work_request", "target_date"]
df_notification_subset = df_notification[required_cols].drop_duplicates()

df_greasing_cardanshaft_final = df_greasing_cardanshaft.merge(
    df_notification_subset,
    on="workorder_id",
    how="left"
)

cols = df_greasing_cardanshaft_final.columns.tolist()
bogie_cols = [c for c in df_greasing_cardanshaft_final.columns if re.search(r"\bbogie[1-8]\b", c)
]

preferred_order = ["filename", "workorder_id", "functional_location", "work_request", "target_date"]
approval_cols = ["greasing_cardan_shaft.approval_date", "greasing_cardan_shaft.technician_id", "greasing_cardan_shaft.supervisor_id"]

df_greasing_cardanshaft_final = df_greasing_cardanshaft_final[
    preferred_order + bogie_cols + approval_cols
]

df_greasing_cardanshaft_final[bogie_cols] = df_greasing_cardanshaft_final[bogie_cols].replace(r"^\s*$", np.nan, regex=True)

# Convert each bogie column to boolean safely
for col in bogie_cols:
    s = df_greasing_cardanshaft_final[col]

    df_greasing_cardanshaft_final[col] = s.map(lambda x: 
        True if (isinstance(x, str) and x.strip().upper() == "TRUE") else
        False if (isinstance(x, str) and x.strip().upper() == "FALSE") else
        True if x == 1 else
        False if x == 0 else
        x  # leave other values (NaN, numbers, etc.) as-is
    )

# Convert to nullable boolean (optional)
df_greasing_cardanshaft_final[bogie_cols] = df_greasing_cardanshaft_final[bogie_cols].astype("boolean")

# Drop rows only if all bogie columns are NaN
df_greasing_cardanshaft_final = df_greasing_cardanshaft_final.dropna(subset=bogie_cols, how="all")

df_greasing_cardanshaft_final.head()

,filename,workorder_id,functional_location,work_request,target_date,greasing_cardan_shaft.bogie2.long_cardan_shaft.1,greasing_cardan_shaft.bogie2.long_cardan_shaft.2,greasing_cardan_shaft.bogie2.long_cardan_shaft.3,greasing_cardan_shaft.bogie2.short_cardan_shaft.4,greasing_cardan_shaft.bogie2.short_cardan_shaft.5,...,greasing_cardan_shaft.bogie6.short_cardan_shaft.6,greasing_cardan_shaft.bogie7.long_cardan_shaft.1,greasing_cardan_shaft.bogie7.long_cardan_shaft.2,greasing_cardan_shaft.bogie7.long_cardan_shaft.3,greasing_cardan_shaft.bogie7.short_cardan_shaft.4,greasing_cardan_shaft.bogie7.short_cardan_shaft.5,greasing_cardan_shaft.bogie7.short_cardan_shaft.6,greasing_cardan_shaft.approval_date,greasing_cardan_shaft.technician_id,greasing_cardan_shaft.supervisor_id
7,RS_PM_HYL_4000493210.pdf,4000493210,RSV023,HYL,16/10/2022,True,True,True,True,True,...,True,True,True,True,True,True,True,28/10/2022,7276,7252
12,RS_PM_HYL_4000487566.pdf,4000487566,RSV021,HYL,16/09/2022,True,True,True,True,True,...,True,True,True,True,True,True,True,24/09/2022,7202,7196
14,RS_PM_HYL_4000616771.pdf,4000616771,RSV021,HYL,19/07/2024,False,False,False,True,True,...,True,True,True,True,True,True,True,04/08/2024,10492,7192
18,RS_PM_HYL_4000629069.pdf,4000629069,RSV025,HYL,26/09/2024,False,False,False,True,True,...,True,True,True,True,True,True,True,26/09/2024,20824,9196
25,RS_PM_HYL_4000561801.pdf,4000561801,RSV025,HYL,26/10/2023,True,True,True,True,True,...,True,True,True,True,True,True,True,26/10/2022,9435,6196


### Train Start Up Test

In [264]:
df_trainstartup_test = dfs_rsd.get("train_startup_test")

for df in [df_trainstartup_test]:
    if df is not None:
        df.columns = df.columns.str.strip()

df_trainstartup_test = df_trainstartup_test.add_prefix("train_startup_test.")

df_trainstartup_test.rename(columns={
    "train_startup_test.workorder_id": "workorder_id",
    "train_startup_test.filename": "filename",
    "train_startup_test.date": "train_startup_test.approval_date",
    "train_startup_test.checked_by.id": "train_startup_test.technician_id",
    "train_startup_test.checked_by.name": "train_startup_test.technician_name",
    "train_startup_test.approval.supervisor_id": "train_startup_test.supervisor_id",
}, inplace=True)

df_trainstartup_test.head()

,workorder_id,filename,train_startup_test.4_car_train_no,train_startup_test.approval_date,train_startup_test.odometer,train_startup_test.train_startup_checks.all_tractions_available.eca4_1,train_startup_test.train_startup_checks.all_tractions_available.eca1_2,train_startup_test.train_startup_checks.all_tractions_available.ica3_1,train_startup_test.train_startup_checks.all_tractions_available.ica2_2,train_startup_test.train_startup_checks.etcs_signaling_systems_available.eca1,...,train_startup_test.exterior.check_headlight_and_tail_light_for_crack_and_damages.eca1,train_startup_test.exterior.check_apron_door_locked_and_secured_and_ensure_no_alarm_at_hmi_(when_train_stabling_in_depot).eca4,train_startup_test.exterior.check_apron_door_locked_and_secured_and_ensure_no_alarm_at_hmi_(when_train_stabling_in_depot).ica3,train_startup_test.exterior.amber_light_indicator_status.ica3,train_startup_test.exterior.check_apron_door_locked_and_secured_and_ensure_no_alarm_at_hmi_(when_train_stabling_in_depot).eca1,train_startup_test.exterior.check_apron_door_locked_and_secured_and_ensure_no_alarm_at_hmi_(when_train_stabling_in_depot).ica2,train_startup_test.exterior.amber_light_indicator_status.ica2,train_startup_test.technician_id,train_startup_test.technician_name,train_startup_test.stamp_id
0,4000586856,RS_PM_WEK_4000586856.pdf,29,25/02/2024,147854.81,not required,not required,True,True,true,...,true,true,true,true,true,true,true,19475,MUHAMMAD DZULIZHAM MAZLAN,
1,4000464193,RS_PM_MTH_4000464193.pdf,25,12/05/2022,401643.56,not required,not required,true,False,True,...,True,True,True,True,True,True,True,7203,Suhali,
2,4000446287,RS_PM_MTH_4000446287.pdf,07,25/01/2022,273316.97,not required,not required,true,true,true,...,True,True,True,True,True,True,True,7203,SUHAIL,
3,4000558454,RS_PM_WEK_4000558454.pdf,22,09/10/2003,566850.29,not required,not required,true,True,True,...,True,True,true,True,True,true,True,12487,MOHD ASYRAF,
4,4000464732,RS_PM_MTH_4000464732.pdf,27,17/05/2022,22082.64,not required,not required,True,true,True,...,True,True,True,True,True,true,true,7203,SUHAIL,


In [265]:
from collections import defaultdict

cols = df_trainstartup_test.columns.tolist()

CAR_ORDER = ["eca1", "ica2", "ica3", "eca4"]

def extract_category(col):
    """
    Everything after 'train_startup_test.' but before the last '.ecaX / .icaX'
    """
    if not col.startswith("train_startup_test."):
        return "meta"

    parts = col.replace("train_startup_test.", "").split(".")
    return ".".join(parts[:-1])

grouped = defaultdict(list)

for c in cols:
    category = extract_category(c)
    grouped[category].append(c)

def car_sort_key(col):
    for idx, car in enumerate(CAR_ORDER):
        if col.endswith(car):
            return idx
    return 99  # non-car fields go last

for category in grouped:
    grouped[category] = sorted(grouped[category], key=car_sort_key)

ordered_cols = []

# keep metadata first
ordered_cols.extend(sorted(grouped.get("meta", [])))

# then all train_startup_test categories
for cat in sorted(c for c in grouped if c != "meta"):
    ordered_cols.extend(grouped[cat])

df_trainstartup_test = df_trainstartup_test[ordered_cols]

df_trainstartup_test.head()

,filename,workorder_id,train_startup_test.4_car_train_no,train_startup_test.approval_date,train_startup_test.odometer,train_startup_test.technician_id,train_startup_test.technician_name,train_startup_test.stamp_id,train_startup_test.apron_door_status.apron_door1.eca1,train_startup_test.apron_door_status.apron_door1.ica2,...,train_startup_test.train_startup_checks.all_tractions_available.ica3_1,train_startup_test.train_startup_checks.all_tractions_available.ica2_2,train_startup_test.train_startup_checks.all_tractions_available.ica3_2,train_startup_test.train_startup_checks.all_tractions_available.eca4_2,train_startup_test.train_startup_checks.all_tractions_available.eca1_1,train_startup_test.train_startup_checks.all_tractions_available.ica2_1,train_startup_test.train_startup_checks.etcs_signaling_systems_available.eca1,train_startup_test.train_startup_checks.etcs_signaling_systems_available.ica2,train_startup_test.train_startup_checks.etcs_signaling_systems_available.ica3,train_startup_test.train_startup_checks.etcs_signaling_systems_available.eca4
0,RS_PM_WEK_4000586856.pdf,4000586856,29,25/02/2024,147854.81,19475,MUHAMMAD DZULIZHAM MAZLAN,,True,True,...,True,True,True,True,True,True,true,not required,not required,true
1,RS_PM_MTH_4000464193.pdf,4000464193,25,12/05/2022,401643.56,7203,Suhali,,True,True,...,true,False,False,True,True,True,True,not required,not required,True
2,RS_PM_MTH_4000446287.pdf,4000446287,07,25/01/2022,273316.97,7203,SUHAIL,,true,true,...,true,true,true,true,true,true,true,not required,not required,true
3,RS_PM_WEK_4000558454.pdf,4000558454,22,09/10/2003,566850.29,12487,MOHD ASYRAF,,True,True,...,true,True,true,True,True,True,True,not required,not required,True
4,RS_PM_MTH_4000464732.pdf,4000464732,27,17/05/2022,22082.64,7203,SUHAIL,,True,true,...,True,true,true,True,True,true,True,not required,not required,True


In [266]:
excluded_cols = ["filename", "workorder_id", "train_startup_test.approval_date", "train_startup_test.odometer", "train_startup_test.technician_id", "train_startup_test.technician_name", "train_startup_test.stamp_id", "train_startup_test.4_car_train_no"]
data_cols = [c for c in df_trainstartup_test.columns if c not in excluded_cols]

data_cols

['train_startup_test.apron_door_status.apron_door1.eca1',
 'train_startup_test.apron_door_status.apron_door1.ica2',
 'train_startup_test.apron_door_status.apron_door1.ica3',
 'train_startup_test.apron_door_status.apron_door1.eca4',
 'train_startup_test.apron_door_status.apron_door2.eca1',
 'train_startup_test.apron_door_status.apron_door2.ica2',
 'train_startup_test.apron_door_status.apron_door2.ica3',
 'train_startup_test.apron_door_status.apron_door2.eca4',
 'train_startup_test.apron_door_status.apron_door3.eca1',
 'train_startup_test.apron_door_status.apron_door3.ica2',
 'train_startup_test.apron_door_status.apron_door3.ica3',
 'train_startup_test.apron_door_status.apron_door3.eca4',
 'train_startup_test.apron_door_status.apron_door4.eca1',
 'train_startup_test.apron_door_status.apron_door4.ica2',
 'train_startup_test.apron_door_status.apron_door4.ica3',
 'train_startup_test.apron_door_status.apron_door4.eca4',
 'train_startup_test.apron_door_status.apron_door5.eca1',
 'train_startu

In [267]:
def normalize_bool_cell(v):
    # 1) NaN stays NaN
    if pd.isna(v):
        return pd.NA

    s = str(v).strip()
    # 2) empty / whitespace -> NA
    if s == "":
        return pd.NA

    s_lower = s.lower()

    # 3) keep "not required" (any case)
    if s_lower == "not required":
        return "not required"

    # 4) any form of True -> True (boolean)
    if s_lower in {"true", "t", "yes", "y", "1"}:
        return True

    # 5) otherwise -> False (boolean)
    # (includes "false", "no", "0", random text, etc.)
    return False

# Apply only to data columns
df_trainstartup_test[data_cols] = df_trainstartup_test[data_cols].map(normalize_bool_cell)

# Optional: make those columns a nullable boolean/string mix (keeps pd.NA properly)
# (You can skip this if you don't need strict dtypes.)
# df_trainstartup_test[data_cols] = df_trainstartup_test[data_cols].astype("object")
df_trainstartup_test.head()

,filename,workorder_id,train_startup_test.4_car_train_no,train_startup_test.approval_date,train_startup_test.odometer,train_startup_test.technician_id,train_startup_test.technician_name,train_startup_test.stamp_id,train_startup_test.apron_door_status.apron_door1.eca1,train_startup_test.apron_door_status.apron_door1.ica2,...,train_startup_test.train_startup_checks.all_tractions_available.ica3_1,train_startup_test.train_startup_checks.all_tractions_available.ica2_2,train_startup_test.train_startup_checks.all_tractions_available.ica3_2,train_startup_test.train_startup_checks.all_tractions_available.eca4_2,train_startup_test.train_startup_checks.all_tractions_available.eca1_1,train_startup_test.train_startup_checks.all_tractions_available.ica2_1,train_startup_test.train_startup_checks.etcs_signaling_systems_available.eca1,train_startup_test.train_startup_checks.etcs_signaling_systems_available.ica2,train_startup_test.train_startup_checks.etcs_signaling_systems_available.ica3,train_startup_test.train_startup_checks.etcs_signaling_systems_available.eca4
0,RS_PM_WEK_4000586856.pdf,4000586856,29,25/02/2024,147854.81,19475,MUHAMMAD DZULIZHAM MAZLAN,,True,True,...,True,True,True,True,True,True,True,not required,not required,True
1,RS_PM_MTH_4000464193.pdf,4000464193,25,12/05/2022,401643.56,7203,Suhali,,True,True,...,True,False,False,True,True,True,True,not required,not required,True
2,RS_PM_MTH_4000446287.pdf,4000446287,07,25/01/2022,273316.97,7203,SUHAIL,,True,True,...,True,True,True,True,True,True,True,not required,not required,True
3,RS_PM_WEK_4000558454.pdf,4000558454,22,09/10/2003,566850.29,12487,MOHD ASYRAF,,True,True,...,True,True,True,True,True,True,True,not required,not required,True
4,RS_PM_MTH_4000464732.pdf,4000464732,27,17/05/2022,22082.64,7203,SUHAIL,,True,True,...,True,True,True,True,True,True,True,not required,not required,True


### Export into Excel

In [268]:
output_file = "../../output/rsd/rolling_stock_final.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    df_trainstartup_test.to_excel(writer, sheet_name="train_startup_test", index=False)
    df_tyre_pressure_final.to_excel(writer, sheet_name="tyre_pressure", index=False)
    df_tyre_wear_final_replaced.to_excel(writer, sheet_name="tyre_wear", index=False)
    df_airbag_pressure_final_replaced.to_excel(writer, sheet_name="airbag_pressure", index=False)
    df_airbag_pressure_final_replaced.to_excel(writer, sheet_name="airbag_pressure", index=False)
    df_cceb_final.to_excel(writer, sheet_name="cceb", index=False)
    df_air_standup_final.to_excel(writer, sheet_name="air_standup", index=False)
    df_waterponding_final.to_excel(writer, sheet_name="water_ponding", index=False)
    df_cardanshaft_final.to_excel(writer, sheet_name="cardan_shaft", index=False)
    df_greasing_cardanshaft_final.to_excel(writer, sheet_name="greasing_cardan_shaft", index=False)
    
print(f"Saved as: {output_file}")

Saved as: ../../output/rsd/rolling_stock_final.xlsx
